# 🌍 Gaussian Processes in Action: Extrapolating Climate Data

In this coding exercise, we will explore how to predict future CO2 concentration based on past observations using Gaussian Processes (GP).

The NOAA CO2 dataset is a collection of direct atmospheric carbon dioxide measurements provided by the National Oceanic and Atmospheric Administration. We will be working the with monthly mean data made Mauna Loa Observatory.
We will take the observations between 2000–2020 as training set and make predicts up to 2031.




In [ ]:
# @title: Repo setup {display-mode: "form"}
import os

# Public course repo — no token needed.
REPO_OWNER  = "eth-fdd-fs26"
REPO_NAME   = "FDD-WE1-public"
BRANCH_NAME = "main"

def _in_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q -b "$BRANCH_NAME" "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"
    # This notebook lives in the exercises/ folder (next to gaussian_process_ui.py), so run from there.
    os.chdir(f'/content/{REPO_NAME}/exercises')

In [ ]:
!pip install -q plotly==6.8.0 anywidget

In [ ]:
import gaussian_process_ui as ui
from gaussian_process_ui import co2_data

### 1. Introduction

In [ ]:
ui.show_introduction()

In [ ]:
# Explore the CO2 dataset
co2_data.head(10)

## 2. What is a Gaussian Process (GP)?

In [ ]:
ui.show_gp_explanation()

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
import numpy as np

In [ ]:
rbf_kernel = RBF(length_scale=2)
gp_unfitted = GaussianProcessRegressor(kernel=rbf_kernel, alpha=0.1, normalize_y=True)

In [ ]:
ui.show_gp_anatomy(gp_unfitted, X_min=0, X_max=10)

In [ ]:
# Prepare a dummy dataset for fitting the GP model
mean_func = lambda x: x/2 + np.sin(x)  # Define a mean function for the GP
X_train_dummy = np.linspace(0, 10, 5).reshape(-1, 1)
y_train_dummy = mean_func(X_train_dummy.flatten()) + np.random.normal(0, 0.1, 5)
# Initialize and fit a GP model
gp = GaussianProcessRegressor(kernel=rbf_kernel, alpha=0.1, normalize_y=True)
gp.fit(X_train_dummy, y_train_dummy)


In [ ]:
ui.show_gp_anatomy(gp, X_min=0, X_max=10)

### 3. The Naive Approach: Standard RBF Kernel
Let's see what happens if we blindly throw a standard **Radial Basis Function (RBF)** kernel to model the $CO_2$ concentration. The kernel being used is given by
$$
K = \text{ConstantKernel}(\text{constant var}) * \text{RBF}(\text{lengthscale}).
$$

Here, the constant kernel allows us to scale the variance of our prediction. A larger $\text{constant var}$ leads to more unpredictable concentration value.

Play with the lengthscale and constant kernel parameter below. Can you make it accurately predict up to 2031?

In [ ]:
ui.show_rbf_example()

In [ ]:
ui.rbf_quiz_1()

In [ ]:
ui.rbf_quiz_2()

### 4. Engineering A Composite Kernel
Because CO2 data has underlying physics (annual planetary cycles and human-driven trends), we must design a kernel that reflects this structure. An easy yet powerful way to obtain new kernels is to combine existing ones via addition or multiplication.

In [ ]:
ui.show_composite_strategy()

In [ ]:
ui.show_composite_example()

In [ ]:
ui.show_lml_explanation()

### 5. 🎯 Implement the Composite Kernel and Perform Optimization

In [ ]:
from sklearn.gaussian_process.kernels import RBF, ExpSineSquared, WhiteKernel, ConstantKernel

In [ ]:
def build_kernel(trend_var, trend_length, 
                 seasonal_var, seasonal_length, seasonal_periodicity,
                 noise_variance, length_scale_bounds=(5, 100)):
    '''
    🎯 TODO: build a composite kernel as described in section 3
    Use the following kernel components from sklearn.gaussian_process.kernels:
    - Trend: ConstantKernel * RBF
    - Seasonal: ConstantKernel * ExpSineSquared
    - Noise: WhiteKernel
    Parameters:
    - trend_var: input of the ConstantKernel in the trend component
    - trend_length: length scale of the trend component
    - seasonal_var: input of the ConstantKernel in the seasonal component
    - seasonal_length: length scale of the seasonal component
    - seasonal_periodicity: periodicity of the seasonal component
    - noise_variance: input of the WhiteKernel
    - length_scale_bounds: search bounds for RBF length scale (to enforce prior knowledge)
    Returns:
    - k_composite: the composite kernel as a sum of the three components
    '''

    k_trend = ... # a ConstantKernel with value trend_var *  RBF with length_scale=trend_length and length_scale_bounds=length_scale_bounds
    k_seasonal = ... # a ConstantKernel with value seasonal_var * ExpSineSquared with length_scale=seasonal_length and periodicity=seasonal_periodicity
    k_noise = ... # a WhiteKernel with noise_level=noise_variance
    k_composite = ... # the sum of the three kernels above
    
    return k_composite

In [ ]:
ui.show_dashboard(build_kernel_func=build_kernel)

## 6. Interpretations

In [ ]:
ui.show_interpretation()